In [ ]:
# ================================================
# Google Colab Script: Discount & Premium Filter
# FIXED: Correct "last completed market day" logic
#        + Limited to max 20 stocks per list
# ================================================

!pip install yfinance pandas pytz -q

import yfinance as yf
import pandas as pd
from datetime import datetime
import pytz

# ================================================
# YOUR RAW LIST (unchanged)
# ================================================
RAW_LIST = """
A
AA
AAL
AAP
AAPL
ABBV
ABNB
ABT
ACN
ADBE
ADI
ADM
ADP
ADSK
AEP
AES
AFL
AFRM
AGNC
AI
AIG
AKAM
ALB
ALK
ALL
AMAT
AMD
AMGN
AMRN
AMZN
APA
APH
APTV
AVGO
AXP
BA
BABA
BAC
BAX
BBY
BEN
BIDU
BIIB
BITO
BK
BKR
BMY
BP
BSX
BX
BYND
C
CAH
CAT
CB
CCI
CCJ
CCL
CF
CFG
CHWY
CI
CL
CLF
CLX
CMCSA
CME
CNC
CNP
COF
COIN
COP
COST
CPB
CPRI
CRM
CRON
CRWD
CSCO
CSX
CTVA
CVNA
CVS
CVX
CZR
D
DAL
DD
DE
DELL
DHI
DHR
DIA
DIS
DKNG
DLR
DOCU
DOW
DRI
DVN
DXC
EA
EBAY
ED
EEM
EFA
EIX
EL
EMR
ENPH
EOG
EQR
EQT
ETSY
EVRG
EW
EWJ
EWW
EWY
EWZ
EXC
EXPE
F
FANG
FAST
FCX
FDX
FE
FI
FITB
FOXA
FSLR
FTI
FTV
FXE
FXI
GD
GDX
GE
GILD
GLD
GLW
GM
GME
GOOG
GOOGL
GPRO
GPS
GS
HAL
HBAN
HBI
HCA
HD
HIG
HLT
HOG
HOLX
HON
HPE
HPQ
HYG
IBB
IBIT
IBM
ICE
IEF
INTC
IP
IPG
IRM
IVZ
IWM
IYR
JCI
JD
JETS
JNJ
JNPR
JPM
K
KHC
KMI
KO
KR
KRE
KSS
KWEB
LEN
LI
LKQ
LLY
LNC
LOW
LQD
LUMN
LUV
LVS
LYB
LYFT
MA
MAR
MARA
MAS
MCD
MCHP
MDLZ
MDT
MET
META
MGM
MMM
MO
MOS
MPC
MRK
MRNA
MRVL
MS
MSFT
MSTR
MTB
MTCH
MU
NCLH
NEE
NEM
NET
NFLX
NKE
NOW
NRG
NTAP
NTES
NVAX
NVDA
NWL
NWSA
O
OIH
OKE
OMC
ON
ORCL
OXY
PARA
PBR
PDD
PENN
PEP
PFE
PFG
PG
PGR
PINS
PLTR
PNC
PPL
PRU
PSX
PTON
PYPL
QCOM
QQQ
RBLX
RCL
RF
RIG
RIOT
RIVN
ROKU
ROST
RTX
RUN
SBUX
SCHD
SCHW
SEDG
SHOP
SIRI
SLB
SLV
SMCI
SMH
SNAP
SNOW
SO
SOFI
SOXL
SOXS
SPG
SPX
SPXL
SPXS
SPY
SQQQ
STX
SWKS
SYF
SYY
T
TAP
TBT
TCOM
TDOC
TFC
TGT
TJX
TLT
TMO
TMUS
TPR
TQQQ
TRIP
TRV
TSLA
TSM
TSN
TT
TTD
TTWO
TXN
U
UA
UAA
UAL
UBER
ULTA
UNG
UNH
UNP
UPS
UPST
URBN
USB
USO
UVXY
V
VALE
VFC
VGK
VTR
VXX
VZ
WAB
WBA
WBD
WDC
WFC
WM
WMB
WMT
WU
WY
WYNN
X
XBI
XEL
XHB
XLB
XLC
XLE
XLF
XLI
XLK
XLP
XLRE
XLU
XLV
XLY
XOM
XOP
XRT
XRX
XSP
XYZ
YELP
YINN
ZION
ZM
"""

tickers = [line.strip() for line in RAW_LIST.strip().split("\n") if line.strip()]

discount_list = []
premium_list = []

print(f"Processing {len(tickers)} tickers...\n")

ny_tz = pytz.timezone('America/New_York')
now_ny = datetime.now(ny_tz)

for ticker in tickers:
    try:
        stock = yf.Ticker(ticker)
        # Get enough history to be safe
        hist = stock.history(period="5d", interval="1d")
        if len(hist) < 1:
            continue

        # ========================================
        # FIXED: Correctly identify the LAST COMPLETED market day
        # ========================================
        last_bar_date = hist.index[-1].tz_convert(ny_tz).date()
        today_ny = now_ny.date()

        market_open_time = now_ny.replace(hour=9, minute=30, second=0, microsecond=0)
        market_close_time = now_ny.replace(hour=16, minute=0, second=0, microsecond=0)

        # If we are currently during regular trading hours AND the last bar is today → it's partial
        if (last_bar_date == today_ny) and (market_open_time <= now_ny < market_close_time):
            # Use the previous completed day
            if len(hist) < 2:
                continue
            prev_day = hist.iloc[-2]
        else:
            # Market is closed / after hours / weekend → last bar is already the completed day
            prev_day = hist.iloc[-1]

        prev_low = float(prev_day['Low'])
        prev_high = float(prev_day['High'])
        prev_range = prev_high - prev_low

        if prev_range <= 0:
            continue

        # Current (live) price
        try:
            info = stock.info
            curr_price = (
                info.get('currentPrice') or
                info.get('regularMarketPrice') or
                info.get('previousClose') or
                float(hist['Close'].iloc[-1])
            )
        except:
            curr_price = float(hist['Close'].iloc[-1])
        curr_price = float(curr_price)

        threshold = 0.05 * prev_range

        # Discount: within bottom 5% of the previous day's range
        if curr_price <= (prev_low + threshold):
            discount_list.append(ticker)

        # Premium: within top 5% of the previous day's range
        if curr_price >= (prev_high - threshold):
            premium_list.append(ticker)

    except:
        continue

# ================================================
# LIMIT TO MAX 20 PER CATEGORY (in original ticker order)
# ================================================
discount_list = discount_list[:20]
premium_list = premium_list[:20]

# ================================================
# RESULTS
# ================================================
print("✅ Processing complete!\n")

print("DISCOUNT LIST (near daily low - bottom 5% of previous day's range):")
print(','.join(discount_list) if discount_list else "(None)")

print("\nPREMIUM LIST (near daily high - top 5% of previous day's range):")
print(','.join(premium_list) if premium_list else "(None)")

print(f"\nSummary: {len(discount_list)} discount stocks | {len(premium_list)} premium stocks (capped at 20 each)")